# 00. Диагностика без обучения

Этот ноутбук ничего не обучает. Он отвечает на четыре вопроса, от которых зависит весь план:

1. Совмещены ли пары LQ–HQ с субпиксельной точностью (иначе потери с фазой учат шуму)?
2. Сколько высоких частот теряет сам VAE (потолок для любого дообучения LoRA)?
3. Что происходит с точностью при росте `lambda_sem` (можно ли «накрутить» VoL/GT без пользы)?
4. Чего модели не хватает больше — амплитуды или фазы?

Запустите его первым. На T4 занимает около получаса (оценка, не замер).

## Инструкция

1. **Настройки ноутбука (панель справа).** Accelerator — `GPU T4 x2`, Internet — `On`, Persistence — `Files only`.
2. **Подключите данные (Add Input):**
   - датасет с кодом — папка `PiSA-SR-main` целиком, вместе с `freq_exp`;
   - обучающие пары LQ/HQ и валидационные пары LQ/HQ;
   - Stable Diffusion 2.1-base, релизный чекпоинт `pisa_sr.pkl` и, если используете теги, `ram_swin_large_14m.pth`.
3. **Ячейка «Пути».** Впишите свои пути. Ячейка «Проверка» сразу скажет, чего не хватает.
4. **Запускайте ячейки по порядку.** Обучение печатает строку каждые 20 шагов и валидацию каждые `EVAL_EVERY` шагов (по умолчанию 500).
5. **Время.** Сессия Kaggle ограничена; если этап не успевает, уменьшите `STEPS` или запускайте этапы по отдельности
   (см. раздел «Если сессия оборвалась» в конце).
6. **Сохраните результат.** Последняя ячейка упаковывает логи и чекпоинты в архив. Нажмите `Save Version` →
   `Save & Run All` либо скачайте архив из `/kaggle/working`. Для сравнения экспериментов
   (ноутбук `08_compare_all`) подключите сохранённые результаты как входные данные.

**Что смотреть в результатах**

- `curve.png` и `curve_*.csv` — LPIPS, VoL/GT и ошибка в высокочастотных полосах по шагам;
- `hf_err` — главная метрика: 1 = детали не восстановлены, больше 1 = выдуманы, меньше — лучше;
- `eff_gsd_cm` — эффективное разрешение в сантиметрах (меньше — лучше);
- `train_log_*.csv` — значение, вес и норма градиента каждого члена потерь. Если норма градиента частотного
  члена меньше 5 % от главного (L2 в pix-этапе, LPIPS в sem-этапе), член почти не влияет — увеличьте вес.

In [ ]:
NOTEBOOK_TAG = "00_diagnostics"

# ==== Пути: отредактируйте под свои датасеты ====
CODE_SRC  = "/kaggle/input/pisa-sr-main/PiSA-SR-main"      # папка с кодом (внутри должна быть freq_exp)
SD_PATH   = "/kaggle/working/sd_v2_1_full"                  # Stable Diffusion 2.1-base
INIT_CKPT = "/kaggle/input/pisa-sr-weights/pisa_sr.pkl"     # релизный чекпоинт PiSA-SR
RAM_PATH  = ""                                              # ram_swin_large_14m.pth; пусто = без RAM
TRAIN_LQ  = "/kaggle/input/hq-lq-netherlands-train/lq_images"
TRAIN_HQ  = "/kaggle/input/hq-lq-netherlands-train/hq_images"
VAL_LQ    = "/kaggle/input/datasets/vende14/hq-lq-netherlands-val/lq_images"
VAL_HQ    = "/kaggle/input/datasets/vende14/hq-lq-netherlands-val/hq_images"

# ==== Параметры обучения ====
OUT_ROOT = "/kaggle/working/experiments/freq"   # сюда пишутся логи и чекпоинты
STEPS    = 4000    # шагов на этап (как в ваших прежних запусках)
EVAL_EVERY = 500   # валидация каждые столько шагов; чаще — точнее кривая, но дольше обучение
ACCUM    = 1       # накопление градиента: 4 сглаживает кривые, но за те же шаги даёт вчетверо меньше обновлений
EMA      = 0       # 0.999 включает EMA весов LoRA (сохраняется отдельный *_ema.pkl)
FDL_CROP = 256     # окно для FDL: 256 экономит память; 0 = весь кроп 512
AUTO     = 0.3     # калибровка весов частотных членов по норме градиента; 0 = взять веса как есть

# Чекпоинт для вопросов 3 и 4: релизный или ваш дообученный (например, pisasr_final.pkl из прежних запусков)
CKPT_TO_CHECK = INIT_CKPT
N_DIAG = 100

In [ ]:
import os, shutil, subprocess, sys

# --- Проверка ---
!nvidia-smi --query-gpu=name,memory.total --format=csv
missing = [p for p in [CODE_SRC, SD_PATH, INIT_CKPT, TRAIN_LQ, TRAIN_HQ, VAL_LQ, VAL_HQ] if not os.path.exists(p)]
assert not missing, f"Не найдены пути: {missing}"
assert os.path.isdir(os.path.join(CODE_SRC, "freq_exp")), "В датасете с кодом нет папки freq_exp"
if RAM_PATH and not os.path.exists(RAM_PATH):
    raise FileNotFoundError(RAM_PATH)

# --- Код копируется в рабочую папку (входные датасеты доступны только для чтения) ---
WORK = "/kaggle/working/PiSA-SR-main"
if not os.path.exists(WORK):
    shutil.copytree(CODE_SRC, WORK)
os.chdir(WORK)

# --- Пакеты: две функции потерь из статей + метрики ---
!pip install -q focal-frequency-loss fdl-pytorch lpips pyiqa
# Если окружение чистое, раскомментируйте установку зависимостей PiSA-SR:
# !pip install -q -r requirements.txt

os.environ.update(SD=SD_PATH, INIT=INIT_CKPT, RAM=RAM_PATH, TRAIN_LQ=TRAIN_LQ, TRAIN_HQ=TRAIN_HQ,
                  VAL_LQ=VAL_LQ, VAL_HQ=VAL_HQ, OUT=OUT_ROOT, STEPS=str(STEPS), ACCUM=str(ACCUM),
                  EMA=str(EMA), FDL_CROP=str(FDL_CROP), AUTO=str(AUTO), EVAL_EVERY=str(EVAL_EVERY))
print("Рабочая папка:", os.getcwd())
!bash freq_exp/experiments.sh list

## 1. Совмещение пар LQ–HQ

Средний сдвиг должен быть меньше 0,25 пикселя HQ.

In [ ]:
!python freq_exp/diag_alignment.py --lq_dir {TRAIN_LQ} --hq_dir {TRAIN_HQ} --n 200
!python freq_exp/diag_alignment.py --lq_dir {VAL_LQ} --hq_dir {VAL_HQ} --n 200

## 2. Потолок VAE

Эталон кодируется и декодируется без UNet. Если ошибка в ВЧ-полосах близка к 1, узкое место — VAE, и потери на UNet дадут ограниченный эффект.

In [ ]:
!python freq_exp/diag_vae_ceiling.py --pretrained_model_path {SD_PATH} --hq_dir {VAL_HQ} --n {N_DIAG}

## 3. Перебор `lambda_sem`

Если с ростом `lambda_sem` отношение `vol_ratio` идёт к 1, а `hf_err` растёт выше 1, резкость добавляется выдуманными деталями.

In [ ]:
!python freq_exp/infer_eval.py --pretrained_model_path {SD_PATH} --ckpt {CKPT_TO_CHECK} \
    --lq_dir {VAL_LQ} --hq_dir {VAL_HQ} --out_dir /kaggle/working/sr/diag --n {N_DIAG} \
    --lambda_sem 0 0.5 1.0 1.2 1.5 2.0

## 4. Амплитуда или фаза

Тест подмены из TFDSR для режима `lambda_sem = 1`.

In [ ]:
!python freq_exp/diag_swap_test.py --sr_dir /kaggle/working/sr/diag/lsem_1.0 --hq_dir {VAL_HQ}

## Что делать с результатами

- Сдвиг пар больше 0,25 пикселя — сначала пересоздать LQ, иначе эксперименты с FFL и Fourier Space Loss бессмысленны.
- Потолок VAE низкий — в дипломе это нужно показать отдельной строкой таблицы; ожидания от потерь снизить.
- Основная доля ВЧ-ошибки от фазы — приоритет у ноутбука `03_V1`; от амплитуды — у `02_K1`.